# AICE Professional 주피터라이트 실전 모의고사 01회 — 군 PBL 합성데이터(차량 사고 보험료 할증 · 사고보고서 · 도면 결함)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 군 차량 사고 보험료 할증 예측 | F1-score 0.92 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 50분 |
| 2. Text | 차량 사고보고서 사고유형 분류 | 정확도 0.95 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 60분 |
| 3. Image | 도면 결함 유형 분류 | 정확도 0.93 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, Keras(TensorFlow.js) | 60분 |

In [1]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 군 차량 사고 보험료 할증 예측 (30점)

<sub>학습 목표: 군 차량 사고 보험료 할증 예측 모델로 테스트 F1-score 0.92 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(StratifiedKFold), LightGBM, joblib | 난이도: 중급 | 예상 시간: 50분</sub>

### 문제 개요

군 차량 사고 기록(부대·지역·차량 유형·연식·운전자 경력·기상·시간대·사고 유형·등급·수리비용·손해율·정비이력)으로 그 사고 때문에 **차년도 보험료가 할증될지** 를 예측하는 이진 분류 문제입니다(부대 수송 안전 진단).

제공된 훈련 데이터셋을 활용해 이진 분류 모델을 개발하고, 테스트 데이터셋의 '보험료할증여부' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **F1-score 0.92 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('보험료할증여부') 값이 제외되어 있습니다.

**Label 정보**

- '보험료할증여부' : 1 = 할증, 0 = 할증 없음 (할증 약 23%)

**Data 정보**

- 'id' : 사고번호
- '사고발생일' : 사고 날짜(YYYY-MM-DD)
- '부대코드 · 지역코드' : 부대·지역
- '차량유형 · 차량연식' : 차량 종류·연식(년)
- '운전자경력연수' : 운전 경력(년)
- '기상상태 · 시간대' : 날씨·주야간
- '사고유형 · 사고등급' : 사고 원인 분류·피해 등급
- '수리비용 · 손해율퍼센트 · 정비이력건수' : 수리비(원)·손해율(%)·정비 이력 수
- '반복사고여부' : 같은 유형 반복 사고 여부(여/부)

**데이터 출처** — 제7회 데이터한마당 본선 1팀(종합보급창) 군 차량 사고·보험 **합성데이터** 1,500건 — 실제 부대 자료 아님

### 데이터 예시

`01_train.csv`

| id | 사고발생일 | 부대코드 | 지역코드 | 차량유형 | 차량연식 | 운전자경력연수 | 기상상태 | … |
|---|---|---|---|---|---|---|---|---|
| ACC-01191 | 2024-10-26 | C부대 | 지역2 | 오토바이 | 6 | 30.1 | 맑음 | … |
| ACC-00455 | 2024-10-18 | J부대 | 지역3 | 버스 | 5 | 12.8 | 비 | … |
| ACC-00373 | 2025-11-11 | D부대 | 지역3 | 버스 | 14 | 27.9 | 맑음 | … |

`01_test_x.csv`

| id | 사고발생일 | 부대코드 | 지역코드 | 차량유형 | 차량연식 | 운전자경력연수 | 기상상태 | … |
|---|---|---|---|---|---|---|---|---|
| ACC-00913 | 2024-06-09 | D부대 | 지역6 | 세단 | 7 | 28.4 | 맑음 | … |
| ACC-01336 | 2025-04-10 | G부대 | 지역4 | 특수차량 | 2 | 5.9 | 맑음 | … |
| ACC-00445 | 2024-09-04 | J부대 | 지역2 | 특수차량 | 6 | 30.9 | 맑음 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 군 차량 사고 보험료 할증 예측 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('보험료할증여부') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 '보험료할증여부' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 '보험료할증여부' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- `info()` 로 자료형을 보고, 날짜는 월·요일 같은 파생 변수로 바꿔 보세요. `id` 는 학습에 쓰지 않습니다.
- 문자열 범주(부대·지역·차량유형·기상·사고유형 등)는 원-핫 인코딩이 필요합니다.
- 할증이 약 23% 로 적습니다. 평가 지표는 **F1-score(할증=1 기준)** 입니다. 예측값은 0/1 정수로 제출하세요.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **F1-score 0.828 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [7]:
import pandas as pd
import joblib
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)
LABEL = '보험료할증여부'

def feats(df):
  X = df.drop(columns = ['id', LABEL], errors = 'ignore').copy()
  d = pd.to_datetime(X.pop('사고발생일'))
  X['월'], X['요일'] = d.dt.month, d.dt.dayofweek
  X['수리비손해율'] = X['수리비용'] / 1e6 * X['손해율퍼센트'] / 100
  return pd.get_dummies(X, dtype=int)

X_train = feats(train)
y_train = train[LABEL]

print(X_train.shape)
print(y_train.shape)

X_train.head()

X_test = feats(test)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1)
pred_y = cross_val_predict(model, X_train, y_train, cv=cv)
score = f1_score(y_train, pred_y)
print('f1_score:', score)

pred_proba = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
  pred = (pred_proba >= threshold).astype(int)
  score = f1_score(y_train, pred)
  print(threshold, score)

model.fit(X_train, y_train)
joblib.dump(model, 'answer_1.pkl')
print('모델 저장 완료')

pred_test = model.predict(X_test)
result = test.copy()
result[LABEL] = pred_test
result.to_csv('answer_1.csv', index=False)

print(result.shape)
result.head()

answer = pd.read_csv('answer_1.csv')

print('데이터 크기: ', answer.shape)
print('마지막 열: ', answer.columns[-1])
print('예측값 분포:')
print(answer[LABEL].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Columns: 16 entries, id to 보험료할증여부
dtypes: float64(1), int64(5), object(10)
memory usage: 150.1+ KB
(1200, 49)
(1200,)
f1_score: 0.9616724738675958
0.1 0.9569707401032702
0.2 0.9601386481802426
0.3 0.96
0.4 0.96
0.5 0.9616724738675958
0.6 0.9540636042402827
0.7 0.9540636042402827
0.8 0.9540636042402827
0.9 0.9464285714285714
모델 저장 완료
(300, 16)
데이터 크기:  (300, 16)
마지막 열:  보험료할증여부
예측값 분포:
보험료할증여부
0    232
1     68
Name: count, dtype: int64


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

자료형·결측·양성(1) 비율부터 확인합니다. 번호·날짜 원문·정답과 같은 정보(누수) 열은 빼고, 범주형은 원-핫 인코딩합니다. '참고'의 파생 변수를 만든 뒤 후보 모델 두세 개를 **교차검증 F1** 로 비교하고, 양성이 적으면 **분류 임계값**도 교차검증으로 고르세요.

- 이 문항의 핵심: `info()` 로 자료형을 보고, 날짜는 월·요일 같은 파생 변수로 바꿔 보세요. `id` 는 학습에 쓰지 않습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_predict`, `f1_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____
from sklearn.metrics import ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)
LABEL = '____'

def feats(df):
    X = df.drop(columns=['____', LABEL], errors='____').copy()
    d = pd.to_datetime(X.pop('____'))
    X['____'], X['____'] = d.dt.month, d.dt.dayofweek
    X['____'] = X['____'] / 1e6 * X['____'] / ____
    return pd.get_dummies(X, dtype=int)

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 분류 임계값을 교차검증으로 고르기
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 차량 사고보고서 사고유형 분류 (35점)

<sub>학습 목표: 차량 사고보고서 사고유형 분류 모델로 테스트 정확도 0.95 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

군 차량 **사고보고서 문장**(날씨·차량·운행 상황·경위·조치)으로 **사고유형** 을 분류하는 자연어 처리 문제입니다. 같은 원인도 여러 표현으로 쓰여 있고, 원인과 무관한 문구(보고·첨부·이력)가 섞여 있습니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(6종) 모델을 개발하고, 테스트 데이터셋의 '사고유형' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 95% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('사고유형') 값이 제외되어 있습니다.

**Label 정보**

- '사고유형' : 과속 / 기타 / 부주의 / 빙판길미끄러짐 / 추돌 / 후진사고

**Data 정보**

- 'id' : 테스트 행 번호(테스트에만)
- '사고보고서' : 사고 경위 보고 문장

**데이터 출처** — 제7회 데이터한마당 본선 1팀 차량 사고보고서 **합성 문장** 1,500건 — 실제 보고서 아님

### 데이터 예시

`02_train.csv`

| 사고보고서 | 사고유형 |
|---|---|
| 영하의 날씨에 주간에 오토바이 운행 중 결빙 구간에서 바퀴가 헛돌며 미끄러짐. 블랙박스 영상 확인 부대 안전담당에게 보고함 | 빙판길미끄러짐 |
| 맑은 날 주간에 버스 운행 중 도로 위 낙하물을 피하지 못하였음. 부대 안전담당에게 보고함 운전자는 경상 과거에 주차장에서 뒤로 빼다가 기둥에 부딪힘 이력 있음. | 기타 |
| 맑은 날 주간에 세단 운행 중 내비게이션 조작 중 방향을 놓침. 보험사 접수 완료 블랙박스 영상 확인 | 부주의 |

`02_test_x.csv`

| id | 사고보고서 |
|---|---|
| 1 | 비가 내리던 중 주간에 세단 운행 중 졸음으로 핸들 조작이 늦어짐. 보험사 접수 완료 목격자 진술 확보 |
| 2 | 화창한 오후 주간에 오토바이 운행 중 후방 확인 없이 물러나다 담장 충격. 보험사 접수 완료 |
| 3 | 짙은 안개 속에서 주간에 오토바이 운행 중 동승자와 대화하다 표지판을 못 봄. 정비반에서 견인 운전자는 경상 |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 차량 사고보고서 사고유형 분류 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('사고유형') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 '사고유형' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 '사고유형' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 형태소 분석기 없이도 **문자 n-gram**(`analyzer='char_wb'`) TF-IDF 가 잘 맞습니다.
- '보고함'·'첨부함' 처럼 모든 보고서에 나오는 문구는 구분에 도움이 안 됩니다(`max_df`·불용어로 줄여 보세요).
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트 행 순서(`id`)를 그대로 유지해 제출하세요.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 85.5% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [9]:
# 답안 작성

In [18]:
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
TEXT, LABEL = '사고보고서', '사고유형'
print(train[LABEL].value_counts().to_dict(), '평균 문장 길이:', round(train[TEXT].str.len().mean(), 2))

X = train[TEXT]
y = train[LABEL]
X_train, X_valid, y_train, y_valid= train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape)
print(X_valid.shape)

model = make_pipeline(TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5)), LinearSVC(random_state=42))
model.fit(X_train, y_train)

pred_y = model.predict(X_valid)
score = accuracy_score(y_valid, pred_y)
print('accuracy:', score)

model.fit(X, y)
joblib.dump(model, 'answer_2.pkl')
print('모델 저장 완료')

pred_test = model.predict(test[TEXT])

result = test.copy()
result[LABEL] = pred_test

result.to_csv('answer_2.csv', index=False)

print(result.shape)
result.head()

answer = pd.read_csv('answer_2.csv')

print('데이터 크기: ', answer.shape)
print('마지막 열: ', answer.columns[-1])
print('예측값 분포:')
print(answer[LABEL].value_counts())



{'추돌': 277, '부주의': 226, '과속': 210, '후진사고': 171, '빙판길미끄러짐': 170, '기타': 146} 평균 문장 길이: 66.52
(960,)
(240,)
accuracy: 0.9875
모델 저장 완료
(300, 3)
데이터 크기:  (300, 3)
마지막 열:  사고유형
예측값 분포:
사고유형
추돌         69
부주의        56
과속         53
후진사고       43
빙판길미끄러짐    43
기타         36
Name: count, dtype: int64


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 형태소 분석기 없이도 **문자 n-gram**(`analyzer='char_wb'`) TF-IDF 가 잘 맞습니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `make_pipeline`, `CountVectorizer`, `TfidfVectorizer`, `MultinomialNB`, `LinearSVC`, `train_test_split`, `accuracy_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
from sklearn.pipeline import ____
from sklearn.feature_extraction.text import ____, ____
from sklearn.naive_bayes import ____
from sklearn.svm import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
TEXT, LABEL = '____', '____'
print(train[LABEL].value_counts().to_dict(), '____', round(train[TEXT].str.len().mean(), ____))


# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 도면 결함 유형 분류 (35점)

<sub>학습 목표: 도면 결함 유형 분류 모델로 테스트 정확도 0.93 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, Keras(TensorFlow.js) | 난이도: 중급 | 예상 시간: 60분</sub>

### 문제 개요

납품 도면을 찍은 이미지(64×64 컬러)로 **도면 결함 유형 6종** 을 분류하는 문제입니다(도면 판독 품질검사 자동화). 결함 표시가 일부만 있거나, 결함과 무관한 눈금·잡점이 섞여 있습니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(6종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 93% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/<클래스>/dwg_*.png`(학습, 클래스별 폴더)와 `03_test/test_*.png`(테스트)가 생깁니다. 64×64 컬러 PNG.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : tolerance: 공차 표시 오류 / dimension: 도면 치수 표기 누락 / weld: 용접부 상세도 미비 / ok: 이상 없음 / material: 재질 규격 불일치 / surface: 표면처리 요구사항 누락

**데이터 출처** — 제7회 데이터한마당 본선 3팀(도면 판독 품질검사) 이미지 형식으로 v3 에서 다시 **합성** — 64×64 컬러, 결함마다 300장

### 데이터 예시

학습 이미지 라벨별 장수: material 240, dimension 240, ok 240, surface 240, weld 240, tolerance 240

| file_name | label |
|---|---|
| dwg_00001.png | material |
| dwg_00002.png | dimension |
| dwg_00003.png | ok |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 도면 결함 유형 분류 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.h5** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 클래스별 폴더 구조이므로 `image_dataset_from_directory` 로 바로 읽을 수 있습니다(`validation_split`·`subset`·`seed`).
- 색(빨강 공차 표시·초록 빗금·갈색 점)과 선의 끊김이 단서입니다. 픽셀 값은 1/255 로 나눠 0~1 로 맞추세요.
- 평가 지표는 **정확도(Accuracy)** 입니다.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 83.7% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [31]:
import os
import zipfile

IMG_DIR = os.environ.get('AICE_IMG_DIR', 'tmp')
if not os.path.exists(os.path.join(IMG_DIR, '03_train')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)

import pandas as pd
from tensorflow.keras.utils import image_dataset_from_directory, load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

test = pd.read_csv('03_test_x.csv')
SIZE, MODE = 64, 'rgb'


kw = dict(
    image_size=(SIZE, SIZE),
    batch_size=32,
    validation_split=0.2,
    seed=42,
    color_mode=MODE
)

tr_ds = image_dataset_from_directory(
    f'{IMG_DIR}/03_train',
    subset='training',
    **kw
)

va_ds = image_dataset_from_directory(
    f'{IMG_DIR}/03_train',
    subset='validation',
    **kw
)

CLASSES = tr_ds.class_names


print('클래스 이름:', CLASSES)
print('클래스 개수:', len(CLASSES))

tr_ds = tr_ds.map(lambda images, labels: (images / 255.0, labels))
va_ds = va_ds.map(lambda images, labels: (images / 255.0, labels))

model = Sequential([
    Input(shape=(SIZE, SIZE, 3)),

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(CLASSES), activation='softmax')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

estop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    tr_ds,
    validation_data=va_ds,
    epochs=25,
    callbacks=[estop]
)

print('최종 학습 정확도:', history.history['accuracy'][-1])
print('최종 검증 정확도:', history.history['val_accuracy'][-1])
print('최고 검증 정확도:', max(history.history['val_accuracy']))

loss, acc = model.evaluate(va_ds, verbose=0)

print('현재 모델 검증 정확도:', acc)
print('현재 모델 검증 손실:', loss)

import numpy as np

model.save('answer_3.h5')
print('모델 저장 완료')

images = np.stack([
    img_to_array(
        load_img(
            os.path.join(IMG_DIR, '03_test', fname),
            target_size=(SIZE, SIZE),
            color_mode=MODE
        )
    )
    for fname in test['file_name']
]) / 255.0

pred_proba = model.predict(images)
pred_idx = np.argmax(pred_proba, axis=1)
pred_test = np.array(CLASSES)[pred_idx]

result = test.copy()
result['label'] = pred_test
result.to_csv('answer_3.csv', index=False)

print('데이터 크기:', result.shape)
print('마지막 열:', result.columns[-1])
print('예측값 분포:')
print(result['label'].value_counts())



Found 1440 files belonging to 6 classes.
Using 1152 files for training.
Found 1440 files belonging to 6 classes.
Using 288 files for validation.
클래스 이름: ['dimension', 'material', 'ok', 'surface', 'tolerance', 'weld']
클래스 개수: 6


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,614 (1.48 MB)

 Trainable params: 388,614 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 242ms/step - accuracy: 0.1936 - loss: 1.7977 - val_accuracy: 0.3056 - val_loss: 1.7693
Epoch 2/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 242ms/step - accuracy: 0.3420 - loss: 1.5549 - val_accuracy: 0.3889 - val_loss: 1.4028
Epoch 3/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 225ms/step - accuracy: 0.4592 - loss: 1.2480 - val_accuracy: 0.5486 - val_loss: 1.1316
Epoch 4/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 8s 209ms/step - accuracy: 0.5894 - loss: 1.0174 - val_accuracy: 0.6701 - val_loss: 0.9079
Epoch 5/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 9s 237ms/step - accuracy: 0.6615 - loss: 0.7942 - val_accuracy: 0.6944 - val_loss: 0.7347
Epoch 6/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 195ms/step - accuracy: 0.7231 - loss: 0.6564 - val_accuracy: 0.7292 - val_loss: 0.5996
Epoch 7/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 8s 236ms/step - accuracy: 0.7639 - loss: 0.5687 - val_accuracy: 0.7708 - val_loss: 0.5132
Epoch 8/25
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 198ms/step - accuracy: 0.8030 - loss: 0.4528 - val_accuracy:

현재 모델 검증 정확도: 0.9722222089767456
현재 모델 검증 손실: 0.11122841387987137
모델 저장 완료
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step
데이터 크기: (360, 2)
마지막 열: label
예측값 분포:
label
ok           62
tolerance    60
dimension    60
surface      60
material     60
weld         58
Name: count, dtype: int64


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 클래스별 폴더 구조이므로 `image_dataset_from_directory` 로 바로 읽을 수 있습니다(`validation_split`·`subset`·`seed`).

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `tensorflow`, `pandas`, `image_dataset_from_directory`, `load_img`, `img_to_array`, `Sequential`, `load_model`, `Input`, `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`, `EarlyStopping`.

- 참고 문서: <https://keras.io/api/data_loading/image/>
- 참고 문서: <https://keras.io/api/data_loading/image/#loadimg-function>
- 참고 문서: <https://keras.io/api/data_loading/image/#imgtoarray-function>
- 참고 문서: <https://keras.io/api/models/sequential/>
- 참고 문서: <https://keras.io/api/layers/convolution_layers/convolution2d/>
- 참고 문서: <https://keras.io/api/layers/pooling_layers/max_pooling2d/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
import ____
from tensorflow.keras.utils import ____, ____, ____
from tensorflow.keras.models import ____, ____
from tensorflow.keras.layers import ____, ____, ____, ____, ____, ____
from tensorflow.keras.callbacks import ____, ____, ____

test = pd.read_csv('03_test_x.csv')
SIZE, MODE = ____, '____'
kw = dict(image_size=(SIZE, SIZE), batch_size=____, validation_split=____, seed=____, color_mode=MODE)
tr_ds = image_dataset_from_directory(f'____', subset='____', **kw)
va_ds = image_dataset_from_directory(f'____', subset='____', **kw)
CLASSES = tr_ds.class_names                               # 폴더 이름(알파벳 순) = 라벨 번호 순서


# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.h5`(모델) · `answer_3.csv`(예측)

</details>

## 제출할 파일 목록

| 문항 | 모델 파일 | 예측 파일 |
|---|---|---|
| 1. Tabular | answer_1.pkl (다른 확장자 가능) | answer_1.csv |
| 2. Text | answer_2.pkl (다른 확장자 가능) | answer_2.csv |
| 3. Image | answer_3.h5 (다른 확장자 가능) | answer_3.csv |
| 공통 | 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장) | |

(주의) 제출 파일명은 **고정**입니다. 위 표의 이름 그대로 저장하세요(앞부분 `answer_` · 문항 번호 · 확장자).
**예시 : answer_1.pkl , answer_1.csv** (이름을 바꾸거나 공백·대쉬(-)를 넣으면 채점되지 않습니다)

## 안내사항

- 이 모의고사는 브라우저 주피터라이트에서 미리 설치된 라이브러리만 씁니다(맨 위 '사용할 수 있는 라이브러리' 표).
- 모델 저장시 본인이 사용하는 프레임워크의 모델 저장 확장자를 사용하셔도 됩니다.(h5, keras, pkl, pickle 등)

In [32]:
# 전체 자동 채점 — 세 문항의 제출 파일(answer_1~3.csv·모델 파일)을 만든 뒤 이 셀을 실행하세요.
# 정답(테스트 라벨)은 압축·인코딩되어 있어 보이지 않습니다. 문항별 지표·통과 여부와 총점을 출력합니다.
PREFIX = 'answer'                                  # 제출 파일명 앞부분(고정)

import base64, json, os, re, zlib
import numpy as np
import pandas as pd

_KEY = "IoCfAgsRhhpKpHHIPcu+1PnsJf+SSzl2mwYUyIIMHur3rpMIRXAAAPN67nUYUUzj2OWSvqQZyQa1lgS8gceTR65/zYeWtJYmaYWWhEQC4asqaa5d0682v6PbT1fRBT0vn4BHXAsAFCy3OE7gwSQCVva2rK2ipnjhZEPFB7+Fdampwba0knB+XCRAXQRz6LfOuY4ViKQsSqC/zreu8auxb3UjAySHq62hhOqAa4Lwnoei38VMOSG3M/0OtXwTZVZ45cAAtfZNKaH/08wH0AlVPayKlYOwRq+cipsPfiViFoW6UN2z2IZMPXMkdl1v3WP91UE/3HkEMdFt703Sy3nTrng7LsxNZuPG6RrizRnlP6E80TNwlSHdA6DGpb7MvOsr4uCo9VsvzlyO8W36preXmbcPpK8wJiv8rEeNqaw4jYE1LMXU/qGuwIf9jcEvxfiBv42to78f4zamv2KlK2br8kZxRUU0tamtRZDtprQJrKuepsyno2ZkYydC1+jF+baXK6y3fKUr8NCWMZ4Frtx4F/gH6muQ6CzMmlXV4gg11WzKKfkFuS0dILpEjTOm2QwZnR9bnWwLBDv/7y/evdYmidn3cNxbwfMMN8EXjV6WTO8QaGBKgG9OXFqBLDZYHCsTj0C3TBOdEsMAbhfTiIp5ExfLamVqdmRpeFSKTSiP3jqRvPuLF/oQADusQb11D9xzn6BtGRP5Ozg+Zc+aRgxo7Jzi+a4wj2tzVWqN2rBsZ+cfDGrveUh3KjLKbCuJ+92QP8gUgKZI75rMlr4uRTAxPLY+PXHUeJB2DCPLLTwUvWtEWf6nAySGvcwRFMtL4zYOOP5Tqd6EoMVQP3vpxrLVM9YmSigVmVATKO6ybeJ6KeLtikVNRPJ64R4CEWEomY+AHx5NBR4WBMJtSgcmWQuQrnktupUFHi7BTUuHfenR0i5CoOtBI4/o9zq6HRNq1pAb6E014cmjuxQ/+sGgK7rcV5MNJOKfwTa28w/4ggzJDuzeUc6bPini6jucjqCAXXKZHTfg+GSdnkwC8WKTm1SQKk4apuo5kEjSdwpkz257Z3UdMgqmws3Z+hZWnDP/Qv5WqkDJWE9+UL/TXtkagnhHS8NEyBu7/kcgXVTLEMvjGn/8OBlzlqvcfsvaV8OzXWinUjSqBgyP01q4+h80kvEH17KrsN3TSXhN+3EKeazXDQXznogqwJ9vkHwOpQJLjE4dBigSwsA7jX54dHoiivrLG034dRcMwkmWWDS22F6yfm4vkOlg/dto6t6ym0+IxynS2LZqbw4ZVTlegJZyiTPYAa6v09LgxpsHVlc8xfL029LrVV5mcEI41LjSQ9LUIjIBHPwawMM5+hd/OAV4svwZGLGlb0DBnufe8DScbDtkjTHw+z6tECy5KrmqdQaYiYA="


def _key():
    raw = base64.b64decode(_KEY)
    return json.loads(zlib.decompress(bytes(b ^ 0x5A for b in raw)).decode("utf-8"))


def _metric(m, y, p):
    if m == "rmse":
        return float(np.sqrt(np.mean((np.asarray(p, float) - np.asarray(y, float)) ** 2)))
    y, p = np.asarray(y).astype(str), np.asarray(p).astype(str)
    if m == "accuracy":
        return float((y == p).mean())
    labels = sorted(set(y)) if m == "macro_f1" else ["1"]
    f1s = []
    for c in labels:
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s))


def grade(prefix="answer", folder="."):
    key = _key()
    phone = prefix                                      # 제출 파일명 앞부분(고정: answer_N.csv · answer_N.pkl/.h5)
    total, rows = 0, []
    for q in key["problems"]:
        no, pts = q["no"], q["points"]
        csv = os.path.join(folder, f"{phone}_{no}.csv")
        models = [f for f in os.listdir(folder) if f.startswith(f"{phone}_{no}.") and not f.endswith(".csv")]
        msg, score, val = [], 0, None
        if not models:
            msg.append("모델 파일 없음")
        if not os.path.exists(csv):
            rows.append((no, pts, 0, None, "예측 CSV 없음")); continue
        sub = pd.read_csv(csv)
        test = pd.read_csv(os.path.join(folder, q["test_file"]))
        lab = q["label"]
        if list(sub.columns) != list(test.columns) + [lab]:
            msg.append(f"열 구조가 다름(테스트 열 + 마지막 '{lab}' 열이어야 함)")
        if lab not in sub.columns or len(sub) != len(test):
            rows.append((no, pts, 0, None, f"행 수 {len(sub)} ≠ {len(test)} 또는 '{lab}' 열 없음")); continue
        keycol = q["key_col"]
        if keycol and not (sub[keycol].astype(str).values == test[keycol].astype(str).values).all():
            msg.append(f"'{keycol}' 순서가 테스트 파일과 다름")
        if sub[lab].isna().any():
            msg.append("예측값에 결측 있음"); sub[lab] = sub[lab].fillna(0 if q["metric"] == "rmse" else "")
        val = _metric(q["metric"], q["y"], sub[lab].values)
        ok = val <= q["pass_line"] if q["metric"] == "rmse" else val >= q["pass_line"]
        if ok and not msg:
            score = pts
        rows.append((no, pts, score, val, "; ".join(msg) or ("통과" if ok else "기준선 미달")))
        total += score
    print(f"{'문항':<6}{'배점':>4}{'점수':>6}  {'지표':<10}{'결과':>10}  {'목표':>8}  {'기준선':>8}  비고")
    for (no, pts, sc, val, note), q in zip(rows, key["problems"]):
        v = "-" if val is None else (f"{val:.4f}")
        print(f"{no:<6}{pts:>4}{sc:>6}  {q['metric']:<10}{v:>10}  {q['target']:>8}  {q['pass_line']:>8}  {note}")
    print(f"총점: {total} / 100  ({'합격' if total >= 80 else '불합격'}, 합격 기준 80점)")
    return total


_점수 = grade(prefix=PREFIX)

문항      배점    점수  지표                결과        목표       기준선  비고
1       30    30  f1            0.9565      0.92     0.828  통과
2       35    35  accuracy      1.0000      0.95     0.855  통과
3       35    35  accuracy      0.9667      0.93     0.837  통과
총점: 100 / 100  (합격, 합격 기준 80점)
